# Lipschitz constant of Swish

In [3]:
import torch

def swish_prime(x, beta=1.0):
    """
    Derivative of Swish: d/dx [x * sigmoid(beta*x)]
    = sigmoid(beta*x) + x * beta * sigmoid(beta*x) * (1 - sigmoid(beta*x))
    """
    s = torch.sigmoid(beta * x)
    return s + x * beta * s * (1.0 - s)

def lipschitz_swish_numeric(
    beta,
    xmin = -12.0,
    xmax = 12.0,
    steps = 1_000_000,      # grid resolution
    batch_size = 200_000,   
):
    """
    Numerically estimates sup_x |swish'(x)| over [xmin, xmax].
    For elementwise activations used on vectors, this is the global Lipschitz constant.
    Returns (L_est, x_at_max).
    """
    
    K_max = torch.tensor(0.0, dtype=torch.float64)
    x_at_max = torch.tensor(float("nan"), dtype=torch.float64)

    # We iterate in ranges to avoid holding a giant tensor
    start = 0
    while start < steps:
        end = min(start + batch_size, steps)
        # +1 so the final overall grid includes xmax exactly when start==0 and end==steps
        xs = torch.linspace(
            xmin + (xmax - xmin) * (start / (steps - 1)),
            xmin + (xmax - xmin) * ((end - 1) / (steps - 1)),
            end - start,
            dtype=torch.float64,
        )
        deriv = swish_prime(xs, beta=beta).abs()
        local_max, idx = torch.max(deriv, dim=0)
        if local_max > K_max:
            K_max = local_max
            x_at_max = xs[idx]
        start = end

    return K_max.item(), x_at_max.item()


K, x_star = lipschitz_swish_numeric(
        beta=1.0,
        xmin=-12.0, xmax=12.0,
        steps=1_000_001,      # odd number so 0 is on the grid
        batch_size=250_000,
    )

print(f"Estimated Lipschitz constant (beta=1): {K:.9f} at x = {x_star:.6f}")

Estimated Lipschitz constant (beta=1): 1.099839320 at x = 2.399352
